# 07 · External and internal fragmentation

Part of **[Inference Engineering](https://github.com/supriyo100/kv-cache-attention-variants)**, an original notebook (not from the upstream course).
Derivation: path step 6. Sources: Kwon et al., SOSP 2023.

Every number below is produced by code in this notebook or in `inference_lab` (tested in
`tests/`). Labels: *measured* = ran here (CPU), *theoretical* = formula with datasheet inputs.

In [ ]:
# Setup: works locally (uv sync) and on Colab (installs the small inference_lab package only)
try:
    import inference_lab  # noqa: F401
except ImportError:  # Colab / fresh environment
    %pip install -q --no-deps "git+https://github.com/supriyo100/kv-cache-attention-variants"
import torch
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (7.5, 3.6), "axes.spines.top": False, "axes.spines.right": False})
C = {"compute": "#0E7C72", "memory": "#B5790A", "waste": "#B42B3E", "shared": "#5B47C9", "ink": "#1C232B"}

## Objective
Measure reservation waste, external fragmentation and internal fragmentation by replaying one
request trace through a contiguous allocator and a paged allocator.

## Theory and equations
* Reservation: a contiguous allocator reserves $T_p + N_{\max}$; mean use is $(T_p + N/2)/(T_p+N_{\max})$.
* External: free memory exists, but no hole is large enough.
* Internal: $w(T) = (-T) \bmod B < B$, mean $(B-1)/2$ if lengths are uniform mod $B$.

## Diagram
```
contiguous: [A....reserved....][hole][B..reserved..][ hole ][C......reserved......]
paged:      [A][A][a.][B][B][B][b..][C][c...][free][free][free]   (lower-case = partial last block)
```

## Implementation

In [ ]:
from inference_lab import fragmentation as F
trace = F.make_trace(300, seed=1)
print(trace[:3])

## Experiment (*measured*, deterministic CPU simulation)

In [ ]:
res = {m: F.simulate(trace, 64 * 1024, m) for m in ("contiguous", "paged")}
for m, s in res.items():
    print(f"{m:10s} admitted={s.admitted:3d}  rejected_external={s.rejected_external:3d}  "
          f"rejected_full={s.rejected_full:3d}  utilization={s.mean_utilization:.1%}  "
          f"waste={s.mean_reservation_waste + s.mean_internal_waste:,.0f} slots")

## Results: internal waste vs block size

In [ ]:
bs = [1, 2, 4, 8, 16, 32, 64, 128]
util = [F.simulate(trace, 64 * 1024, "paged", block_size=b).mean_utilization for b in bs]
fig, ax = plt.subplots()
ax.plot(bs, [100 * (1 - u) for u in util], "o-", color=C["waste"])
ax.set_xscale("log", base=2); ax.set_xlabel("block size (tokens)"); ax.set_ylabel("internal waste (% of allocated)")
plt.show()
print([f"{b}:{100*(1-u):.2f}%" for b, u in zip(bs, util)])

## Interpretation
The contiguous allocator rejects more than half the trace while memory is free in total. That
is external fragmentation. Paging removes it and turns reservation waste into a sub-block
remainder. Internal waste grows roughly linearly with block size and stays small up to 16–32.

## Limitations
No preemption (paged mode stops a request early if the pool is dry) and no compaction in the
contiguous allocator. The trace is synthetic.

## Conclusion
Paging fixes the waste that scales with sequence length. What remains is bounded by the block
size.